# 第 9 章代码实验：内存管理、复制与垃圾回收

观察题请按以下顺序：先阅读代码，不运行；把 prediction 字典中整个 `"__未填写__"`（包括引号）替换成预测的 Python 值；运行本题代码；在 observation 字典中填写实际结果；立即运行本题检查；在解释区比较预测和观察。

预测错误时保留最初预测，只纠正实际观察并解释原因。检查器确认预测已填写，但不要求最初预测正确；它只核对实际观察与实现行为，不能证明你已经理解。

每个字段的两位编号对应一个表达式或状态，中文描述不提示结果。实验 22～24 是独立实现题：先在方案区写思路，再替换占位定义。

回收实验用弱引用观察目标，不输出或长期保存目标本身。以普通 CPython 3.12.13 为验证基线，不要求固定地址、总引用数、代数或精确回收数量。实验 6 在独立子进程调整 GC 开关，不改变当前内核开关。运行实验会实际产生观察输出，交付的 Notebook 不预存这些结果。


## 实验 1：赋值没有复制列表

预测名字的身份关系与最终状态。逐行区分修改共享对象和重新绑定名字。

### 运行前预测

先填写下一单元格，再执行实验代码。

In [ ]:
prediction_01 = {
    "01. alias 重新绑定前 alias is original": "__未填写__",
    "02. append 后 original 的快照": "__未填写__",
    "03. 重新绑定后 original 的内容": "__未填写__",
    "04. 重新绑定后 alias 的内容": "__未填写__",
}


### 运行并观察

现在运行本题代码，将实际结果与最初预测对照。

In [ ]:
original = [1, 2]
alias = original
same_before = alias is original
alias.append(3)
after_append = original.copy()
alias = [9]
print(same_before, after_append, original, alias)


### 实际观察记录

填写刚才实际看到的结果；保留前面的预测，差异在原因区说明。

In [ ]:
observation_01 = {
    "01. alias 重新绑定前 alias is original": "__未填写__",
    "02. append 后 original 的快照": "__未填写__",
    "03. 重新绑定后 original 的内容": "__未填写__",
    "04. 重新绑定后 alias 的内容": "__未填写__",
}


### 实验 1 即时检查

只检查本题，不依赖其他题是否已经填写。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 1)


### 实验 1 原因解释

哪一行修改了共享列表，哪一行只改了 alias 的绑定？请画出两次赋值后的对象图。

请在本单元格或个人笔记中写出自己的解释；预测有误时说明原来漏看了哪条引用或哪个阶段。

## 实验 2：删除名字后容器仍持有实例

先预测删除调用处名字的结果，再判断清空容器后的目标状态。只观察弱引用，不额外保存其返回对象。

### 运行前预测

先填写下一单元格，再执行实验代码。

In [ ]:
prediction_02 = {
    "01. 删除 payload 后 reference() 是否仍能取得目标": "__未填写__",
    "02. 清空 holders 并请求回收后 reference() 是否为 None": "__未填写__",
}


### 运行并观察

现在运行本题代码，将实际结果与最初预测对照。

In [ ]:
import gc
import weakref

class Payload:
    pass

payload = Payload()
reference = weakref.ref(payload)
holders = [payload]
del payload
gc.collect()
alive_with_holder = reference() is not None
holders.clear()
gc.collect()
gone_after_clear = reference() is None
print(alive_with_holder, gone_after_clear)


### 实际观察记录

填写刚才实际看到的结果；保留前面的预测，差异在原因区说明。

In [ ]:
observation_02 = {
    "01. 删除 payload 后 reference() 是否仍能取得目标": "__未填写__",
    "02. 清空 holders 并请求回收后 reference() 是否为 None": "__未填写__",
}


### 实验 2 即时检查

只检查本题，不依赖其他题是否已经填写。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 2)


### 实验 2 原因解释

del 的目标是哪一个名字？容器中的强引用何时消失？这里为什么不把 reference() 的返回值长期保存？

请在本单元格或个人笔记中写出自己的解释；预测有误时说明原来漏看了哪条引用或哪个阶段。

## 实验 3：引用计数的相对变化

本题限定普通 CPython 3.12 构建，与当前学习环境一致；其他构建应阅读主讲义边界后单独设计观察。预测相对变化，不填写绝对计数。

### 运行前预测

先填写下一单元格，再执行实验代码。

In [ ]:
prediction_03 = {
    "01. 增加 alias 后的计数差": "__未填写__",
    "02. 再放入 holders 后的计数差": "__未填写__",
    "03. 移走 alias 和容器引用后计数是否回到 before": "__未填写__",
}


### 运行并观察

现在运行本题代码，将实际结果与最初预测对照。

In [ ]:
import platform
import sys

if platform.python_implementation() != "CPython" or sys.version_info[:2] != (3, 12):
    raise RuntimeError("本题使用普通 CPython 3.12 的观察基线")

class Payload:
    pass

payload = Payload()
before = sys.getrefcount(payload)
alias = payload
with_alias = sys.getrefcount(payload)
holders = [payload]
with_container = sys.getrefcount(payload)
del alias
holders.clear()
after = sys.getrefcount(payload)
print(with_alias - before, with_container - with_alias, after == before)


### 实际观察记录

填写刚才实际看到的结果；保留前面的预测，差异在原因区说明。

In [ ]:
observation_03 = {
    "01. 增加 alias 后的计数差": "__未填写__",
    "02. 再放入 holders 后的计数差": "__未填写__",
    "03. 移走 alias 和容器引用后计数是否回到 before": "__未填写__",
}


### 实验 3 即时检查

只检查本题，不依赖其他题是否已经填写。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 3)


### 实验 3 原因解释

getrefcount 调用自身会怎样影响计数？为什么选择新建实例而不选择 None 或小整数？此实验为什么不能变成跨构建的绝对数字规则？

请在本单元格或个人笔记中写出自己的解释；预测有误时说明原来漏看了哪条引用或哪个阶段。

## 实验 4：函数返回后对象仍可存活

函数只把对象交给调用者，并返回一个弱引用观察句柄。预测函数退出与调用者释放的区别。

### 运行前预测

先填写下一单元格，再执行实验代码。

In [ ]:
prediction_04 = {
    "01. 函数返回后 reference() is result": "__未填写__",
    "02. 调用者释放 result 并回收后 reference() 是否为 None": "__未填写__",
}


### 运行并观察

现在运行本题代码，将实际结果与最初预测对照。

In [ ]:
import gc
import weakref

class Payload:
    pass

def build():
    local = Payload()
    return local, weakref.ref(local)

result, reference = build()
alive_after_return = reference() is result
del result
gc.collect()
gone_after_caller_release = reference() is None
print(alive_after_return, gone_after_caller_release)


### 实际观察记录

填写刚才实际看到的结果；保留前面的预测，差异在原因区说明。

In [ ]:
observation_04 = {
    "01. 函数返回后 reference() is result": "__未填写__",
    "02. 调用者释放 result 并回收后 reference() 是否为 None": "__未填写__",
}


### 实验 4 即时检查

只检查本题，不依赖其他题是否已经填写。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 4)


### 实验 4 原因解释

局部名字结束和对象生命周期结束为什么不是同一件事？返回前后由谁持有强引用？

请在本单元格或个人笔记中写出自己的解释；预测有误时说明原来漏看了哪条引用或哪个阶段。

## 实验 5：删除属性与删除容器位置

同一个目标先由属性和列表共同持有。逐步预测移走每条引用后的状态。

### 运行前预测

先填写下一单元格，再执行实验代码。

In [ ]:
prediction_05 = {
    "01. 删除 holder.value 后目标是否仍存活": "__未填写__",
    "02. 再删除 items[0] 并回收后目标是否已不可取得": "__未填写__",
}


### 运行并观察

现在运行本题代码，将实际结果与最初预测对照。

In [ ]:
import gc
import weakref

class Box:
    pass

target = Box()
holder = Box()
holder.value = target
items = [target]
reference = weakref.ref(target)
del target
del holder.value
gc.collect()
alive_after_attribute = reference() is not None
del items[0]
gc.collect()
gone_after_slot = reference() is None
print(alive_after_attribute, gone_after_slot)


### 实际观察记录

填写刚才实际看到的结果；保留前面的预测，差异在原因区说明。

In [ ]:
observation_05 = {
    "01. 删除 holder.value 后目标是否仍存活": "__未填写__",
    "02. 再删除 items[0] 并回收后目标是否已不可取得": "__未填写__",
}


### 实验 5 即时检查

只检查本题，不依赖其他题是否已经填写。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 5)


### 实验 5 原因解释

三条 del 各自删除的是名字、属性还是容器位置？解释为什么前两条不够。

请在本单元格或个人笔记中写出自己的解释；预测有误时说明原来漏看了哪条引用或哪个阶段。

## 实验 6：自动循环回收开关与手动回收

下面启动当前解释器的独立子进程，子进程中的 GC 开关会恢复，不改变 Notebook 内核开关。先阅读子进程源码并预测三个关系。不要预测收集数量。

### 运行前预测

先填写下一单元格，再执行实验代码。

In [ ]:
prediction_06 = {
    "01. 子进程手动 collect 前目标是否可取得": "__未填写__",
    "02. 子进程手动 collect 后目标是否为 None": "__未填写__",
    "03. 手动 collect 后自动开关是否仍关闭": "__未填写__",
}


### 运行并观察

现在运行本题代码，将实际结果与最初预测对照。

In [ ]:
import json
import subprocess
import sys

source = r"""
import gc
import json
import weakref

class Node:
    pass

was_enabled = gc.isenabled()
gc.disable()
try:
    node = Node()
    node.peer = node
    reference = weakref.ref(node)
    del node
    before_collect = reference() is not None
    gc.collect()
    after_collect = reference() is None
    automatic_still_off = not gc.isenabled()
    print(json.dumps([before_collect, after_collect, automatic_still_off]))
finally:
    if was_enabled:
        gc.enable()
"""
result = subprocess.run([sys.executable, "-c", source], check=True, capture_output=True, text=True)
observed_states = json.loads(result.stdout)
print(observed_states)


### 实际观察记录

填写刚才实际看到的结果；保留前面的预测，差异在原因区说明。

In [ ]:
observation_06 = {
    "01. 子进程手动 collect 前目标是否可取得": "__未填写__",
    "02. 子进程手动 collect 后目标是否为 None": "__未填写__",
    "03. 手动 collect 后自动开关是否仍关闭": "__未填写__",
}


### 实验 6 即时检查

只检查本题，不依赖其他题是否已经填写。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 6)


### 实验 6 原因解释

环内的引用为什么让普通引用计数无法单独解决它？手动收集和自动触发分别由什么决定？为什么隔离到子进程？

请在本单元格或个人笔记中写出自己的解释；预测有误时说明原来漏看了哪条引用或哪个阶段。

## 实验 7：外部强引用对循环对象回收的影响

容器仍持有自引用对象时请求回收，然后移走容器引用再次请求回收。比较两次目标状态。

### 运行前预测

先填写下一单元格，再执行实验代码。

In [ ]:
prediction_07 = {
    "01. roots 仍持有节点时节点是否存活": "__未填写__",
    "02. roots 清空并回收后节点是否已不可取得": "__未填写__",
}


### 运行并观察

现在运行本题代码，将实际结果与最初预测对照。

In [ ]:
import gc
import weakref

class Node:
    pass

node = Node()
node.peer = node
roots = [node]
reference = weakref.ref(node)
del node
gc.collect()
alive_with_root = reference() is not None
roots.clear()
gc.collect()
gone_without_root = reference() is None
print(alive_with_root, gone_without_root)


### 实际观察记录

填写刚才实际看到的结果；保留前面的预测，差异在原因区说明。

In [ ]:
observation_07 = {
    "01. roots 仍持有节点时节点是否存活": "__未填写__",
    "02. roots 清空并回收后节点是否已不可取得": "__未填写__",
}


### 实验 7 即时检查

只检查本题，不依赖其他题是否已经填写。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 7)


### 实验 7 原因解释

哪条路径让第一次收集必须保留整个环？环的形状与垃圾状态有什么不同？

请在本单元格或个人笔记中写出自己的解释；预测有误时说明原来漏看了哪条引用或哪个阶段。

## 实验 8：对象跟踪、回收结果与自动回收开关

只观察接口含义和现场结果，不要求固定阈值、代数或精确回收数量。状态字段要依据表达式判断。

### 运行前预测

先填写下一单元格，再执行实验代码。

In [ ]:
prediction_08 = {
    "01. gc.is_tracked(node) 的结果": "__未填写__",
    "02. gc.collect() 的结果是否为 int": "__未填写__",
    "03. 手动 collect 是否保持原自动开关状态": "__未填写__",
}


### 运行并观察

现在运行本题代码，将实际结果与最初预测对照。

In [ ]:
import gc

class Node:
    pass

node = Node()
enabled_before = gc.isenabled()
tracked = gc.is_tracked(node)
collection_result = gc.collect()
result_is_integer = type(collection_result) is int
enabled_unchanged = gc.isenabled() == enabled_before
print(tracked, result_is_integer, enabled_unchanged)


### 实际观察记录

填写刚才实际看到的结果；保留前面的预测，差异在原因区说明。

In [ ]:
observation_08 = {
    "01. gc.is_tracked(node) 的结果": "__未填写__",
    "02. gc.collect() 的结果是否为 int": "__未填写__",
    "03. 手动 collect 是否保持原自动开关状态": "__未填写__",
}


### 实验 8 即时检查

只检查本题，不依赖其他题是否已经填写。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 8)


### 实验 8 原因解释

被跟踪为什么不表示已不可达？为什么不把 collection_result 当成字节数或固定答案？

请在本单元格或个人笔记中写出自己的解释；预测有误时说明原来漏看了哪条引用或哪个阶段。

## 实验 9：列表浅拷贝的两种修改

先记录外层和内层身份关系，再比较追加内部对象与替换外层位置的影响。

### 运行前预测

先填写下一单元格，再执行实验代码。

In [ ]:
prediction_09 = {
    "01. copied is original": "__未填写__",
    "02. 初始 copied[0] is original[0]": "__未填写__",
    "03. 内层 append 后 original 的快照": "__未填写__",
    "04. 外层替换后 original 的内容": "__未填写__",
    "05. 外层替换后 copied 的内容": "__未填写__",
}


### 运行并观察

现在运行本题代码，将实际结果与最初预测对照。

In [ ]:
import copy

original = [[1], [2]]
copied = copy.copy(original)
outer_same = copied is original
inner_same = copied[0] is original[0]
copied[0].append(9)
after_inner_change = [row.copy() for row in original]
copied[0] = [100]
print(outer_same, inner_same, after_inner_change, original, copied)


### 实际观察记录

填写刚才实际看到的结果；保留前面的预测，差异在原因区说明。

In [ ]:
observation_09 = {
    "01. copied is original": "__未填写__",
    "02. 初始 copied[0] is original[0]": "__未填写__",
    "03. 内层 append 后 original 的快照": "__未填写__",
    "04. 外层替换后 original 的内容": "__未填写__",
    "05. 外层替换后 copied 的内容": "__未填写__",
}


### 实验 9 即时检查

只检查本题，不依赖其他题是否已经填写。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 9)


### 实验 9 原因解释

新外层保存了哪些旧引用？替换外层位置为什么没有改掉另一个外层位置？

请在本单元格或个人笔记中写出自己的解释；预测有误时说明原来漏看了哪条引用或哪个阶段。

## 实验 10：字典浅拷贝与选择性重建

同一源数据建立浅拷贝和只复制必要字段的记录，预测两者对成绩修改的反应。

### 运行前预测

先填写下一单元格，再执行实验代码。

In [ ]:
prediction_10 = {
    "01. original 中是否出现 new": "__未填写__",
    "02. 修改源成绩后 shallow 中的 scores": "__未填写__",
    "03. 修改源成绩后 snapshot 中的 scores": "__未填写__",
}


### 运行并观察

现在运行本题代码，将实际结果与最初预测对照。

In [ ]:
original = {"name": "Ada", "scores": [80]}
shallow = original.copy()
snapshot = {"name": original["name"], "scores": list(original["scores"])}
shallow["new"] = True
original["scores"].append(90)
print("new" in original, shallow["scores"], snapshot["scores"])


### 实际观察记录

填写刚才实际看到的结果；保留前面的预测，差异在原因区说明。

In [ ]:
observation_10 = {
    "01. original 中是否出现 new": "__未填写__",
    "02. 修改源成绩后 shallow 中的 scores": "__未填写__",
    "03. 修改源成绩后 snapshot 中的 scores": "__未填写__",
}


### 实验 10 即时检查

只检查本题，不依赖其他题是否已经填写。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 10)


### 实验 10 原因解释

两个复制方案分别承诺到哪一层？如果成绩元素变成可变字典，旧方案是否仍足够？

请在本单元格或个人笔记中写出自己的解释；预测有误时说明原来漏看了哪条引用或哪个阶段。

## 实验 11：重复引用与逐次创建

比较列表乘法和每轮新建列表，判断各个位置指向谁。

### 运行前预测

先填写下一单元格，再执行实验代码。

In [ ]:
prediction_11 = {
    "01. 修改后 shared_rows 的内容": "__未填写__",
    "02. 修改后 separate_rows 的内容": "__未填写__",
    "03. shared_rows 前两行是否同一对象": "__未填写__",
    "04. separate_rows 前两行是否同一对象": "__未填写__",
}


### 运行并观察

现在运行本题代码，将实际结果与最初预测对照。

In [ ]:
shared_rows = [[]] * 3
separate_rows = [[] for _ in range(3)]
shared_rows[0].append("A")
separate_rows[0].append("A")
print(shared_rows, separate_rows)
print(shared_rows[0] is shared_rows[1], separate_rows[0] is separate_rows[1])


### 实际观察记录

填写刚才实际看到的结果；保留前面的预测，差异在原因区说明。

In [ ]:
observation_11 = {
    "01. 修改后 shared_rows 的内容": "__未填写__",
    "02. 修改后 separate_rows 的内容": "__未填写__",
    "03. shared_rows 前两行是否同一对象": "__未填写__",
    "04. separate_rows 前两行是否同一对象": "__未填写__",
}


### 实验 11 即时检查

只检查本题，不依赖其他题是否已经填写。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 11)


### 实验 11 原因解释

乘法复制的是位置引用还是内部对象本身？怎样避免只根据外层长度猜对象数量？

请在本单元格或个人笔记中写出自己的解释；预测有误时说明原来漏看了哪条引用或哪个阶段。

## 实验 12：深拷贝保留结果内部共享

原数据两个位置共同指向一行。预测副本内部的共享以及原图与副本的隔离。

### 运行前预测

先填写下一单元格，再执行实验代码。

In [ ]:
prediction_12 = {
    "01. copied 两个位置是否同一对象": "__未填写__",
    "02. copied[0] 是否与 original[0] 同一对象": "__未填写__",
    "03. 修改副本后的 original": "__未填写__",
    "04. 修改副本后的 copied": "__未填写__",
}


### 运行并观察

现在运行本题代码，将实际结果与最初预测对照。

In [ ]:
import copy

shared = [1]
original = [shared, shared]
copied = copy.deepcopy(original)
same_inside_copy = copied[0] is copied[1]
shares_with_original = copied[0] is original[0]
copied[0].append(2)
print(same_inside_copy, shares_with_original, original, copied)


### 实际观察记录

填写刚才实际看到的结果；保留前面的预测，差异在原因区说明。

In [ ]:
observation_12 = {
    "01. copied 两个位置是否同一对象": "__未填写__",
    "02. copied[0] 是否与 original[0] 同一对象": "__未填写__",
    "03. 修改副本后的 original": "__未填写__",
    "04. 修改副本后的 copied": "__未填写__",
}


### 实验 12 即时检查

只检查本题，不依赖其他题是否已经填写。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 12)


### 实验 12 原因解释

memo 需要记住什么对应关系？“与原图分离”和“结果所有位置彼此独立”为什么不同？

请在本单元格或个人笔记中写出自己的解释；预测有误时说明原来漏看了哪条引用或哪个阶段。

## 实验 13：深拷贝处理自引用

只判断对象关系，不输出或手工展开无限嵌套。预测副本的第一项究竟指向谁。

### 运行前预测

先填写下一单元格，再执行实验代码。

In [ ]:
prediction_13 = {
    "01. copied is original": "__未填写__",
    "02. copied[0] is copied": "__未填写__",
    "03. len(copied) 的结果": "__未填写__",
}


### 运行并观察

现在运行本题代码，将实际结果与最初预测对照。

In [ ]:
import copy

original = []
original.append(original)
copied = copy.deepcopy(original)
print(copied is original, copied[0] is copied, len(copied))


### 实际观察记录

填写刚才实际看到的结果；保留前面的预测，差异在原因区说明。

In [ ]:
observation_13 = {
    "01. copied is original": "__未填写__",
    "02. copied[0] is copied": "__未填写__",
    "03. len(copied) 的结果": "__未填写__",
}


### 实验 13 即时检查

只检查本题，不依赖其他题是否已经填写。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 13)


### 实验 13 原因解释

如果递归复制时不先登记已处理对象，会遇到什么问题？副本保留了哪一种图形关系？

请在本单元格或个人笔记中写出自己的解释；预测有误时说明原来漏看了哪条引用或哪个阶段。

## 实验 14：一次整体复制与分开复制

两种方案都调用 deepcopy，但复制的边界不同。预测结果内部身份关系与修改后的内容。

### 运行前预测

先填写下一单元格，再执行实验代码。

In [ ]:
prediction_14 = {
    "01. together 内两项是否同一对象": "__未填写__",
    "02. separate 内两项是否同一对象": "__未填写__",
    "03. 修改后 together 的内容": "__未填写__",
    "04. 修改后 separate 的内容": "__未填写__",
}


### 运行并观察

现在运行本题代码，将实际结果与最初预测对照。

In [ ]:
import copy

shared = [1]
original = [shared, shared]
together = copy.deepcopy(original)
separate = [copy.deepcopy(item) for item in original]
together[0].append(2)
separate[0].append(2)
print(together[0] is together[1], separate[0] is separate[1])
print(together, separate)


### 实际观察记录

填写刚才实际看到的结果；保留前面的预测，差异在原因区说明。

In [ ]:
observation_14 = {
    "01. together 内两项是否同一对象": "__未填写__",
    "02. separate 内两项是否同一对象": "__未填写__",
    "03. 修改后 together 的内容": "__未填写__",
    "04. 修改后 separate 的内容": "__未填写__",
}


### 实验 14 即时检查

只检查本题，不依赖其他题是否已经填写。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 14)


### 实验 14 原因解释

哪些调用共享同一次复制过程的备忘录？如果需求就是拆开每一行，共享关系应怎样声明？

请在本单元格或个人笔记中写出自己的解释；预测有误时说明原来漏看了哪条引用或哪个阶段。

## 实验 15：复制函数与复制可变字段

观察函数对象和数据字段在同一次深拷贝中的处理。不要把“深”解释为任何节点都换身份。

### 运行前预测

先填写下一单元格，再执行实验代码。

In [ ]:
prediction_15 = {
    "01. copied is original": "__未填写__",
    "02. copied 中 operation 是否仍为原函数": "__未填写__",
    "03. 两份 values 是否同一列表": "__未填写__",
    "04. 通过副本 operation 调用 3 的结果": "__未填写__",
}


### 运行并观察

现在运行本题代码，将实际结果与最初预测对照。

In [ ]:
import copy

def operation(value):
    return value * 2

original = {"operation": operation, "values": [1, 2]}
copied = copy.deepcopy(original)
print(copied is original)
print(copied["operation"] is operation)
print(copied["values"] is original["values"])
print(copied["operation"](3))


### 实际观察记录

填写刚才实际看到的结果；保留前面的预测，差异在原因区说明。

In [ ]:
observation_15 = {
    "01. copied is original": "__未填写__",
    "02. copied 中 operation 是否仍为原函数": "__未填写__",
    "03. 两份 values 是否同一列表": "__未填写__",
    "04. 通过副本 operation 调用 3 的结果": "__未填写__",
}


### 实验 15 即时检查

只检查本题，不依赖其他题是否已经填写。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 15)


### 实验 15 原因解释

哪一种共享是工具明确定义的行为？为什么不能只比较所有 id 是否变化来验收复制？

请在本单元格或个人笔记中写出自己的解释；预测有误时说明原来漏看了哪条引用或哪个阶段。

## 实验 16：取出弱引用后局部名字又持有目标

预测保存 reference() 返回对象的影响，再移走该局部绑定。

### 运行前预测

先填写下一单元格，再执行实验代码。

In [ ]:
prediction_16 = {
    "01. borrowed 尚在时 reference() is borrowed": "__未填写__",
    "02. 删除 borrowed 并回收后 reference() 是否为 None": "__未填写__",
}


### 运行并观察

现在运行本题代码，将实际结果与最初预测对照。

In [ ]:
import gc
import weakref

class Payload:
    pass

payload = Payload()
reference = weakref.ref(payload)
borrowed = reference()
del payload
gc.collect()
alive_while_borrowed = reference() is borrowed
del borrowed
gc.collect()
gone_after_borrowed = reference() is None
print(alive_while_borrowed, gone_after_borrowed)


### 实际观察记录

填写刚才实际看到的结果；保留前面的预测，差异在原因区说明。

In [ ]:
observation_16 = {
    "01. borrowed 尚在时 reference() is borrowed": "__未填写__",
    "02. 删除 borrowed 并回收后 reference() 是否为 None": "__未填写__",
}


### 实验 16 即时检查

只检查本题，不依赖其他题是否已经填写。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 16)


### 实验 16 原因解释

弱句柄、目标和 borrowed 三者分别是什么？哪条新强引用使目标继续存在？

请在本单元格或个人笔记中写出自己的解释；预测有误时说明原来漏看了哪条引用或哪个阶段。

## 实验 17：弱值表与绑定方法对实例生命周期的影响

同一实例既放入弱值表，又把其绑定方法放入普通列表。预测各阶段结果。

### 运行前预测

先填写下一单元格，再执行实验代码。

In [ ]:
prediction_17 = {
    "01. 回调列表保留方法时 A 是否仍在弱值表": "__未填写__",
    "02. 清空回调列表并回收后 A 是否仍在弱值表": "__未填写__",
}


### 运行并观察

现在运行本题代码，将实际结果与最初预测对照。

In [ ]:
import gc
import weakref

class Listener:
    def notify(self):
        return "ready"

listener = Listener()
table = weakref.WeakValueDictionary()
table["A"] = listener
callbacks = [listener.notify]
del listener
gc.collect()
found_with_callback = "A" in table
callbacks.clear()
gc.collect()
found_after_clear = "A" in table
print(found_with_callback, found_after_clear)


### 实际观察记录

填写刚才实际看到的结果；保留前面的预测，差异在原因区说明。

In [ ]:
observation_17 = {
    "01. 回调列表保留方法时 A 是否仍在弱值表": "__未填写__",
    "02. 清空回调列表并回收后 A 是否仍在弱值表": "__未填写__",
}


### 实验 17 即时检查

只检查本题，不依赖其他题是否已经填写。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 17)


### 实验 17 原因解释

弱值表是否是唯一持有者？绑定方法的 __self__ 怎样参与引用路径？

请在本单元格或个人笔记中写出自己的解释；预测有误时说明原来漏看了哪条引用或哪个阶段。

## 实验 18：闭包延长对象生命周期

读函数保存外层参数，但不实际调用它取得目标。预测删除外部名字和删除闭包后的区别。

### 运行前预测

先填写下一单元格，再执行实验代码。

In [ ]:
prediction_18 = {
    "01. reader 存在时目标是否仍可取得": "__未填写__",
    "02. 删除 reader 并回收后目标是否为 None": "__未填写__",
}


### 运行并观察

现在运行本题代码，将实际结果与最初预测对照。

In [ ]:
import gc
import weakref

class Payload:
    pass

def make_reader(payload):
    def read():
        return payload
    return read

payload = Payload()
reference = weakref.ref(payload)
reader = make_reader(payload)
del payload
gc.collect()
alive_with_reader = reference() is not None
del reader
gc.collect()
gone_after_reader = reference() is None
print(alive_with_reader, gone_after_reader)


### 实际观察记录

填写刚才实际看到的结果；保留前面的预测，差异在原因区说明。

In [ ]:
observation_18 = {
    "01. reader 存在时目标是否仍可取得": "__未填写__",
    "02. 删除 reader 并回收后目标是否为 None": "__未填写__",
}


### 实验 18 即时检查

只检查本题，不依赖其他题是否已经填写。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 18)


### 实验 18 原因解释

外层函数已经返回后，哪一种绑定关系还在？为什么本题不把 reader() 的结果保存起来？

请在本单元格或个人笔记中写出自己的解释；预测有误时说明原来漏看了哪条引用或哪个阶段。

## 实验 19：缓存持有返回对象

只保留缓存与弱观察句柄。预测清理调用处名字与清空缓存各自能结束什么引用。

### 运行前预测

先填写下一单元格，再执行实验代码。

In [ ]:
prediction_19 = {
    "01. 缓存未清空时目标是否存活": "__未填写__",
    "02. 缓存清空并回收后目标是否为 None": "__未填写__",
}


### 运行并观察

现在运行本题代码，将实际结果与最初预测对照。

In [ ]:
import gc
import weakref
from functools import lru_cache

class Payload:
    pass

@lru_cache(maxsize=2)
def build(key):
    return Payload()

result = build("A")
reference = weakref.ref(result)
del result
gc.collect()
alive_in_cache = reference() is not None
build.cache_clear()
gc.collect()
gone_after_clear = reference() is None
print(alive_in_cache, gone_after_clear)


### 实际观察记录

填写刚才实际看到的结果；保留前面的预测，差异在原因区说明。

In [ ]:
observation_19 = {
    "01. 缓存未清空时目标是否存活": "__未填写__",
    "02. 缓存清空并回收后目标是否为 None": "__未填写__",
}


### 实验 19 即时检查

只检查本题，不依赖其他题是否已经填写。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 19)


### 实验 19 原因解释

缓存条目保存的是结果副本还是返回对象？设置容量还需要考虑什么单项大小与失效问题？

请在本单元格或个人笔记中写出自己的解释；预测有误时说明原来漏看了哪条引用或哪个阶段。

## 实验 20：暂停生成器保存局部形参

生成器只产出一个标记，并未把 payload 交给外部。预测暂停期间和明确关闭后的目标状态。

### 运行前预测

先填写下一单元格，再执行实验代码。

In [ ]:
prediction_20 = {
    "01. 第一次 next(generator) 的结果": "__未填写__",
    "02. 生成器暂停时目标是否可取得": "__未填写__",
    "03. 关闭生成器并回收后目标是否为 None": "__未填写__",
}


### 运行并观察

现在运行本题代码，将实际结果与最初预测对照。

In [ ]:
import gc
import weakref

class Payload:
    pass

def produce(payload):
    yield "ready"
    yield payload

payload = Payload()
reference = weakref.ref(payload)
generator = produce(payload)
del payload
marker = next(generator)
gc.collect()
alive_while_paused = reference() is not None
generator.close()
gc.collect()
gone_after_close = reference() is None
print(marker, alive_while_paused, gone_after_close)


### 实际观察记录

填写刚才实际看到的结果；保留前面的预测，差异在原因区说明。

In [ ]:
observation_20 = {
    "01. 第一次 next(generator) 的结果": "__未填写__",
    "02. 生成器暂停时目标是否可取得": "__未填写__",
    "03. 关闭生成器并回收后目标是否为 None": "__未填写__",
}


### 实验 20 即时检查

只检查本题，不依赖其他题是否已经填写。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 20)


### 实验 20 原因解释

哪个局部形参继续持有目标？如果前一条 yield 已经把目标交给外部名字，close 还保证外部对象消失吗？

请在本单元格或个人笔记中写出自己的解释；预测有误时说明原来漏看了哪条引用或哪个阶段。

## 实验 21：终结回调的执行次数与目标引用

回调只保存 events 和字符串，不捕获目标。预测回收后的记录、终结器状态及再次调用的效果。

### 运行前预测

先填写下一单元格，再执行实验代码。

In [ ]:
prediction_21 = {
    "01. 回收后的 events 快照": "__未填写__",
    "02. 回收后 finalizer.alive": "__未填写__",
    "03. 再次调用 finalizer() 后的 events": "__未填写__",
}


### 运行并观察

现在运行本题代码，将实际结果与最初预测对照。

In [ ]:
import gc
import weakref

class Payload:
    pass

events = []
payload = Payload()
finalizer = weakref.finalize(payload, events.append, "released")
del payload
gc.collect()
first_events = events.copy()
alive_after_collection = finalizer.alive
finalizer()
print(first_events, alive_after_collection, events)


### 实际观察记录

填写刚才实际看到的结果；保留前面的预测，差异在原因区说明。

In [ ]:
observation_21 = {
    "01. 回收后的 events 快照": "__未填写__",
    "02. 回收后 finalizer.alive": "__未填写__",
    "03. 再次调用 finalizer() 后的 events": "__未填写__",
}


### 实验 21 即时检查

只检查本题，不依赖其他题是否已经填写。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 21)


### 实验 21 原因解释

为什么这里的回调不会留住 payload？如果换成 payload.close，哪条引用可能破坏预期？为什么终结器仍不能代替确定关闭？

请在本单元格或个人笔记中写出自己的解释；预测有误时说明原来漏看了哪条引用或哪个阶段。

## 实验 22：实现每行独立的两层复制

实现 clone_rows(rows)。rows 是由整数列表组成的可迭代对象，也可能只能遍历一次。返回新的外层列表，每个位置都对应一个新建的行列表；即使输入两个位置共享同一行，结果也必须拆开成独立两行。保留值与顺序，不修改输入。只要求复制这两层，不要求处理任意嵌套对象图。空输入返回空列表。

### 实现前方案

先写出复制边界或所有权，以及你准备检查的正常、空输入和共享状态场景。

In [ ]:
def clone_rows(rows):
    raise NotImplementedError("请实现 clone_rows")


### 自己的运行观察

请添加小例子并记录对象关系、状态和边界结果，不只记录是否报错。

### 实验 22 即时检查

先运行定义单元格，再检查当前实现。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 22)


### 实验 22 原因解释

先画出有重复行引用的输入，再画出题意要求的输出。这个接口与一次整体 deepcopy 保留共享的语义有什么区别？

## 实验 23：实现保留共享和循环引用的深拷贝

实现 snapshot_graph(value)。输入由内置 dict、list、tuple、整数、字符串和 None 组成，可以有重复引用或循环。返回按深拷贝语义构造的结果：原图与结果图不共享可变容器，结果内部保留原来的共享关系和循环关系，且不修改输入。不可变节点允许复用。允许使用标准库 copy，不要求手写复制算法。

### 实现前方案

先写出复制边界或所有权，以及你准备检查的正常、空输入和共享状态场景。

In [ ]:
def snapshot_graph(value):
    raise NotImplementedError("请实现 snapshot_graph")


### 自己的运行观察

请添加小例子并记录对象关系、状态和边界结果，不只记录是否报错。

### 实验 23 即时检查

先运行定义单元格，再检查当前实现。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 23)


### 实验 23 原因解释

一次复制的边界是什么？怎样验证结果既与原图隔离，又保留原图内部的共享与循环？为什么不能用 JSON 代替此接口？

## 实验 24：实现弱引用对象注册表

实现 WeakRegistry。add(name, value) 以字符串名登记支持弱引用的对象；同名覆盖。get(name) 返回当前对象，不存在或已回收时返回 None。remove(name) 移除登记，缺失时也不报错。len(registry) 返回当前条目数。注册表不能单独使值存活；不要修改传入对象。不支持弱引用的值由 add 抛出 TypeError。可以使用标准库弱值字典；本题不要求并发安全。

### 实现前方案

先写出复制边界或所有权，以及你准备检查的正常、空输入和共享状态场景。

In [ ]:
class WeakRegistry:
    def __init__(self):
        raise NotImplementedError("请初始化 WeakRegistry")

    def add(self, name, value):
        raise NotImplementedError("请实现 add")

    def get(self, name):
        raise NotImplementedError("请实现 get")

    def remove(self, name):
        raise NotImplementedError("请实现 remove")

    def __len__(self):
        raise NotImplementedError("请实现 __len__")


### 自己的运行观察

请添加小例子并记录对象关系、状态和边界结果，不只记录是否报错。

### 实验 24 即时检查

先运行定义单元格，再检查当前实现。

In [ ]:
from chapter9_checks import check_experiment

check_experiment(globals(), 24)


### 实验 24 原因解释

注册表持有的是值的哪一种引用？get 的调用方保存返回对象后，谁又成为强引用所有者？这个表与保证命中的缓存有什么区别？

## 全部复查

24 道实验各自完成即时检查后，再运行整章复查。它不代替预测、逐题观察与原因解释。请闭卷说明一个共享图、一个不可达环和一个缓存滞留场景。

In [ ]:
from chapter9_checks import run_checks

run_checks(globals())
